## Aprendizaje supervisado: Modelo de Clasificación

En esta notebook desarrollamos un modelo supervisado de clasificación para categorizar la producción de petróleo del mes siguiente en tres niveles: **Baja**, **Media** y **Alta**.

El objetivo es complementar el modelo de regresión desarrollado en la notebook 04. Mientras que la regresión estima una cantidad concreta de producción en metros cúbicos, la clasificación permite resumir la producción futura en categorías más fáciles de interpretar.

La variable objetivo utilizada es `clase_prod_pet_mes_siguiente`, construida en la notebook 03 a partir de la producción real del mes siguiente.

El flujo de trabajo será:

1. Cargar y verificar los datasets de modelado;
2. Analizar la distribución de las clases;
3. Definir las variables predictoras;
4. Entrenar un modelo de referencia;
5. Entrenar un Random Forest Classifier;
6. Evaluar el desempeño con métricas de clasificación;
7. Validar temporalmente el modelo con pliegues cronológicos y buscar sus hiperparámetros con `GridSearchCV`;
8. Entrenar el clasificador final con la mejor configuración;
9. Evaluar la configuración seleccionada en el conjunto de test;
10. Interpretar los resultados y sus limitaciones.

### 1. Objetivo de clasificación

Cada fila representa un pozo observado en un determinado mes de corte. Las variables predictoras describen la información disponible hasta ese momento y la variable objetivo indica el nivel de producción de petróleo del mes siguiente.

El modelo debe responder la siguiente pregunta:

> ¿La producción de petróleo del próximo mes será baja, media o alta?

Esta formulación no busca predecir el valor exacto de producción. Para ese objetivo utilizamos el modelo de regresión de la notebook 04. En esta notebook evaluamos la capacidad del modelo para asignar correctamente una categoría de producción.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
# Buscamos la carpeta raíz del proyecto a partir de la ubicación
# desde la que se abrió la notebook.
ubicacion = Path.cwd()

ROOT = next(
    (
        ruta
        for ruta in [ubicacion, *ubicacion.parents]
        if (ruta / "data" / "processed").is_dir()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "No encontramos la carpeta data/processed. "
        "Revisá desde qué carpeta se abrió la notebook."
    )

RUTA_PROCESSED = ROOT / "data" / "processed"

print(f"Carpeta del proyecto: {ROOT}")
print(f"Carpeta de datos procesados: {RUTA_PROCESSED}")

Carpeta del proyecto: /Users/julianvarelasilva/Downloads/ypf-data
Carpeta de datos procesados: /Users/julianvarelasilva/Downloads/ypf-data/data/processed


### 2. Carga y verificación de los datasets

Cargamos los conjuntos de entrenamiento, validación y test generados en la notebook 03.

Utilizamos las mismas particiones temporales que en la notebook de regresión:

- entrenamiento: objetivos de febrero a diciembre de 2025;
- validación: objetivos de enero a marzo de 2026;
- test: objetivos de abril a julio de 2026.

En esta primera versión mantenemos las particiones originales para que la comparación con la notebook 04 sea directa. Más adelante podremos evaluar si resulta necesario ampliar también la ventana de entrenamiento de clasificación.

In [3]:
# Archivos generados por la notebook 03.
archivos_requeridos = [
    "train_modelado.csv",
    "validation_modelado.csv",
    "test_modelado.csv",
    "columnas_modelado.json",
]

archivos_faltantes = [
    nombre
    for nombre in archivos_requeridos
    if not (RUTA_PROCESSED / nombre).is_file()
]

if archivos_faltantes:
    raise FileNotFoundError(
        "Faltan archivos de modelado en data/processed: "
        + ", ".join(archivos_faltantes)
        + ". Ejecutá primero la notebook 03."
    )

with open(
    RUTA_PROCESSED / "columnas_modelado.json",
    encoding="utf-8",
) as archivo:
    definicion_modelado = json.load(archivo)

fechas = ["fecha", "mes_objetivo"]

train = pd.read_csv(
    RUTA_PROCESSED / "train_modelado.csv",
    parse_dates=fechas,
    low_memory=False,
)

validation = pd.read_csv(
    RUTA_PROCESSED / "validation_modelado.csv",
    parse_dates=fechas,
    low_memory=False,
)

test = pd.read_csv(
    RUTA_PROCESSED / "test_modelado.csv",
    parse_dates=fechas,
    low_memory=False,
)

print(f"Filas de entrenamiento: {len(train):,}")
print(f"Filas de validación: {len(validation):,}")
print(f"Filas de test: {len(test):,}")

Filas de entrenamiento: 49,452
Filas de validación: 14,320
Filas de test: 19,737


In [4]:
# Recuperamos las variables predictoras definidas en la notebook 03.
features_numericas = definicion_modelado["features_numericas"]
features_booleanas = definicion_modelado["features_booleanas"]
features_categoricas = definicion_modelado["features_categoricas"]

features = (
    features_numericas
    + features_booleanas
    + features_categoricas
)

objetivo_clasificacion = (
    definicion_modelado["target_clasificacion"]
)

objetivo_regresion = (
    definicion_modelado["target_regresion"]
)

if objetivo_clasificacion not in train.columns:
    raise KeyError(
        f"No se encontró el objetivo de clasificación: "
        f"{objetivo_clasificacion}"
    )

columnas_faltantes = sorted(
    set(features) - set(train.columns)
)

if columnas_faltantes:
    raise KeyError(
        "Faltan variables predictoras en el conjunto de entrenamiento: "
        f"{columnas_faltantes}"
    )

print(f"Variables predictoras: {len(features)}")
print(f"Objetivo de clasificación: {objetivo_clasificacion}")
print(f"Objetivo de regresión disponible: {objetivo_regresion}")

Variables predictoras: 41
Objetivo de clasificación: clase_prod_pet_mes_siguiente
Objetivo de regresión disponible: prod_pet_mes_siguiente


#### 2.1. Resumen de las particiones

Antes de entrenar el clasificador verificamos el tamaño, la cantidad de pozos y el período de cada conjunto. Esta revisión permite confirmar que se conserva el orden temporal utilizado en la notebook de regresión.

In [5]:
particiones = {
    "Entrenamiento": train,
    "Validación": validation,
    "Test": test,
}

resumen_particiones = []

for nombre, datos in particiones.items():
    resumen_particiones.append({
        "Conjunto": nombre,
        "Filas": len(datos),
        "Columnas": datos.shape[1],
        "Pozos": datos["idpozo"].nunique(),
        "Corte desde": datos["fecha"].min().strftime("%Y-%m"),
        "Corte hasta": datos["fecha"].max().strftime("%Y-%m"),
        "Objetivo desde": (
            datos["mes_objetivo"].min().strftime("%Y-%m")
        ),
        "Objetivo hasta": (
            datos["mes_objetivo"].max().strftime("%Y-%m")
        ),
    })

display(
    pd.DataFrame(resumen_particiones)
    .set_index("Conjunto")
)

,Filas,Columnas,Pozos,Corte desde,Corte hasta,Objetivo desde,Objetivo hasta
Conjunto,,,,,,,
Entrenamiento,49452,58,4707,2025-01,2025-11,2025-02,2025-12
Validación,14320,58,4814,2025-12,2026-02,2026-01,2026-03
Test,19737,58,5015,2026-03,2026-06,2026-04,2026-07


### 3. Distribución de la variable objetivo

La variable objetivo de clasificación es `clase_prod_pet_mes_siguiente`. Esta variable fue construida en la notebook 03 a partir de la producción real de petróleo del mes siguiente.

Las categorías representan distintos niveles de producción futura:

- **Baja**: producción nula o inferior al primer umbral definido;
- **Media**: producción ubicada entre los umbrales bajo y alto;
- **Alta**: producción superior al umbral correspondiente.

En esta sección analizamos la cantidad y proporción de cada clase en los conjuntos de entrenamiento, validación y test.

Este análisis es necesario para detectar posibles desequilibrios. Si una clase concentra la mayoría de los registros, la accuracy por sí sola puede ser engañosa, por lo que también utilizaremos balanced accuracy, precision, recall y F1.

In [6]:
# Verificamos que el objetivo de clasificación no tenga valores faltantes.
for nombre, datos in particiones.items():
    if datos[objetivo_clasificacion].isna().any():
        raise ValueError(
            f"Hay valores faltantes en el objetivo de clasificación "
            f"del conjunto {nombre}."
        )

# Mostramos la cantidad de registros de cada clase.
distribucion_clases = []

for nombre, datos in particiones.items():
    conteos = datos[objetivo_clasificacion].value_counts(
        dropna=False
    )
    porcentajes = datos[objetivo_clasificacion].value_counts(
        normalize=True,
        dropna=False,
    ) * 100

    for clase in ["Baja", "Media", "Alta"]:
        distribucion_clases.append({
            "Conjunto": nombre,
            "Clase": clase,
            "Registros": int(conteos.get(clase, 0)),
            "Porcentaje (%)": porcentajes.get(clase, 0.0),
        })

distribucion_clases = pd.DataFrame(distribucion_clases)

display(
    distribucion_clases
    .set_index(["Conjunto", "Clase"])
    .round(2)
)

Registros  Porcentaje (%)
Conjunto      Clase                           
Entrenamiento Baja       18316           37.04
              Media      15568           31.48
              Alta       15568           31.48
Validación    Baja        5055           35.30
              Media       4404           30.75
              Alta        4861           33.95
Test          Baja        6906           34.99
              Media       5825           29.51
              Alta        7006           35.50

Además de revisar la distribución, verificamos que el conjunto de entrenamiento contenga las tres clases esperadas. Esta validación evita intentar entrenar un clasificador sin alguna de las categorías que luego necesitamos predecir.

In [7]:
# Confirmamos que el entrenamiento contenga las tres clases esperadas.
clases_esperadas = {"Baja", "Media", "Alta"}
clases_entrenamiento = set(
    train[objetivo_clasificacion].dropna().unique()
)

clases_faltantes = sorted(
    clases_esperadas - clases_entrenamiento
)

if clases_faltantes:
    raise ValueError(
        "Faltan clases en el conjunto de entrenamiento: "
        f"{clases_faltantes}"
    )

print(
    "Clases disponibles en entrenamiento:",
    sorted(clases_entrenamiento),
)

Clases disponibles en entrenamiento: ['Alta', 'Baja', 'Media']


### 4. Preparación de las variables para clasificación

Separamos las variables predictoras de la variable objetivo de clasificación.

Utilizaremos las mismas 41 variables definidas en la notebook 03 y utilizadas en la notebook de regresión. De esta manera, la comparación entre ambos problemas se realiza sobre la misma información disponible al cierre de cada mes.

El objetivo del clasificador será `clase_prod_pet_mes_siguiente`, mientras que las variables identificadoras, las fechas y el objetivo de regresión no se utilizan como predictores.

In [8]:
# Separamos las variables predictoras y el objetivo de clasificación.
X_train = train[features].copy()
X_validation = validation[features].copy()
X_test = test[features].copy()

y_train = train[objetivo_clasificacion].copy()
y_validation = validation[objetivo_clasificacion].copy()
y_test = test[objetivo_clasificacion].copy()

# Convertimos las variables booleanas a valores numéricos.
# False representa 0 y True representa 1.
for columna in features_booleanas:
    X_train[columna] = X_train[columna].astype("float64")
    X_validation[columna] = X_validation[columna].astype("float64")
    X_test[columna] = X_test[columna].astype("float64")

print(f"Filas de entrenamiento: {len(X_train):,}")
print(f"Filas de validación: {len(X_validation):,}")
print(f"Filas de test: {len(X_test):,}")
print(f"Variables predictoras: {X_train.shape[1]}")
print(f"Clases de entrenamiento: {sorted(y_train.unique())}")

Filas de entrenamiento: 49,452
Filas de validación: 14,320
Filas de test: 19,737
Variables predictoras: 41
Clases de entrenamiento: ['Alta', 'Baja', 'Media']


In [9]:
# Confirmamos que validación y test utilicen las mismas clases
# definidas en el entrenamiento.
clases_train = set(y_train.unique())
clases_validation = set(y_validation.unique())
clases_test = set(y_test.unique())

if clases_validation - clases_train:
    raise ValueError(
        "Validación contiene clases ausentes en entrenamiento: "
        f"{sorted(clases_validation - clases_train)}"
    )

if clases_test - clases_train:
    raise ValueError(
        "Test contiene clases ausentes en entrenamiento: "
        f"{sorted(clases_test - clases_train)}"
    )

print("Las clases de validación y test están contempladas en entrenamiento.")

Las clases de validación y test están contempladas en entrenamiento.


### 5. Modelo de referencia

Antes de entrenar un modelo de clasificación, construimos una referencia sencilla que asigna siempre la clase más frecuente del conjunto de entrenamiento.

Esta referencia no aprende relaciones entre las variables predictoras y el objetivo. Su función es establecer un punto de comparación: el Random Forest debería superar este resultado para demostrar que está incorporando información útil de las características de los pozos y de su historial productivo.

In [10]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

# Creamos una referencia que siempre predice la clase más frecuente.
modelo_referencia = DummyClassifier(
    strategy="most_frequent"
)

modelo_referencia.fit(X_train, y_train)

y_pred_referencia = modelo_referencia.predict(X_validation)

resultados_referencia = pd.DataFrame(
    {
        "Accuracy": [
            accuracy_score(
                y_validation,
                y_pred_referencia,
            )
        ],
        "Balanced accuracy": [
            balanced_accuracy_score(
                y_validation,
                y_pred_referencia,
            )
        ],
        "Precision macro": [
            precision_score(
                y_validation,
                y_pred_referencia,
                average="macro",
                zero_division=0,
            )
        ],
        "Recall macro": [
            recall_score(
                y_validation,
                y_pred_referencia,
                average="macro",
                zero_division=0,
            )
        ],
        "F1 macro": [
            f1_score(
                y_validation,
                y_pred_referencia,
                average="macro",
                zero_division=0,
            )
        ],
    },
    index=["Referencia: clase mayoritaria"],
)

display(resultados_referencia.round(3))

,Accuracy,Balanced accuracy,Precision macro,Recall macro,F1 macro
Referencia: clase mayoritaria,0.353,0.333,0.118,0.333,0.174


### 6. Random Forest Classifier inicial

Ahora entrenamos un Random Forest Classifier utilizando las mismas variables predictoras que usamos en el modelo de regresión.

El preprocesamiento se incorpora dentro de un pipeline para garantizar que:

- las variables numéricas reciban un tratamiento adecuado de los valores faltantes;
- las variables categóricas se conviertan a un formato numérico;
- el preprocesamiento se ajuste únicamente con el conjunto de entrenamiento;
- validación y test se utilicen solamente para evaluar.

Utilizamos el mismo tratamiento de faltantes que en la notebook 04:

- **Numéricas:** completamos con la mediana de entrenamiento y agregamos una columna indicadora por cada variable con faltantes (`add_indicator=True`). Así el modelo puede distinguir un valor imputado de un valor observado.
- **Categóricas:** completamos los faltantes con la categoría explícita `"Sin dato"` en lugar de la categoría más frecuente.

Este punto es importante para `tipo_terminacion_ultima`: el 29 % de las filas de entrenamiento no tiene fracturas disponibles, y en esas filas la clase **Baja** representa el 56 % de los registros, contra el 29 % en el resto. Si completáramos con la categoría más frecuente (`Tapón disparo`), esos pozos quedarían mezclados con el grupo de pozos fracturados más numeroso y se perdería una señal útil.

Las categorías con menos de 20 registros en entrenamiento se agrupan en una categoría común (`min_frequency=20`), igual que en la regresión.

En esta primera versión no aplicamos ponderación de clases, porque la distribución observada es relativamente equilibrada entre las categorías Baja, Media y Alta.

In [11]:
# Definimos las transformaciones para cada tipo de variable.
# Usamos el mismo criterio de faltantes que en la notebook 04.
transformador_clasificacion = ColumnTransformer(
    transformers=[
        (
            "numericas",
            Pipeline([
                (
                    "imputacion",
                    SimpleImputer(strategy="median", add_indicator=True),
                ),
            ]),
            features_numericas,
        ),
        (
            "booleanas",
            Pipeline([
                (
                    "imputacion",
                    SimpleImputer(
                        strategy="most_frequent"
                    ),
                ),
            ]),
            features_booleanas,
        ),
        (
            "categoricas",
            Pipeline([
                (
                    "imputacion",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="Sin dato",
                    ),
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        min_frequency=20,
                        # Matriz densa: el Random Forest entrena mucho más
                        # rápido que con una matriz dispersa.
                        sparse_output=False,
                    ),
                ),
            ]),
            features_categoricas,
        ),
    ],
    remainder="drop",
)

# Utilizamos una configuración comparable con el Random Forest
# seleccionado para el modelo de regresión.
modelo_clasificacion = Pipeline([
    (
        "preprocesamiento",
        transformador_clasificacion,
    ),
    (
        "modelo",
        RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            max_features=0.7,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

print("Entrenando el clasificador...")
modelo_clasificacion.fit(X_train, y_train)
print("Entrenamiento finalizado.")

Entrenando el clasificador...


Entrenamiento finalizado.


#### 6.1. Evaluación inicial en validación

Evaluamos el Random Forest Classifier sobre el conjunto de validación, cuyos objetivos corresponden a los meses de enero a marzo de 2026. El modelo fue entrenado previamente utilizando los registros de entrenamiento, correspondientes a objetivos entre febrero y diciembre de 2025.

En esta etapa comparamos el desempeño del clasificador con el modelo de referencia que siempre predice la clase mayoritaria. De esta manera verificamos si el Random Forest logra aprender relaciones útiles a partir de las variables del pozo y de su historial de producción.

Para evaluar las tres categorías utilizamos las siguientes métricas:

- **Accuracy:** proporción de predicciones correctas sobre el total de registros.
- **Balanced accuracy:** promedio del recall obtenido en las tres clases.
- **Precision macro:** promedio de la precisión calculada para cada clase.
- **Recall macro:** promedio de los casos correctamente identificados en cada clase.
- **F1 macro:** promedio del equilibrio entre precision y recall para las tres clases.

Utilizamos las versiones macro de las métricas para otorgar la misma importancia a las clases Baja, Media y Alta, aunque tengan cantidades de registros diferentes.

In [12]:
# Generamos las predicciones del clasificador sobre el conjunto
# de validación, que representa objetivos de enero a marzo de 2026.
y_pred_clasificacion = modelo_clasificacion.predict(
    X_validation
)

# Calculamos las métricas generales del Random Forest.
resultados_rf_validacion = pd.DataFrame(
    {
        "Accuracy": [
            accuracy_score(
                y_validation,
                y_pred_clasificacion,
            )
        ],
        "Balanced accuracy": [
            balanced_accuracy_score(
                y_validation,
                y_pred_clasificacion,
            )
        ],
        "Precision macro": [
            precision_score(
                y_validation,
                y_pred_clasificacion,
                average="macro",
                zero_division=0,
            )
        ],
        "Recall macro": [
            recall_score(
                y_validation,
                y_pred_clasificacion,
                average="macro",
                zero_division=0,
            )
        ],
        "F1 macro": [
            f1_score(
                y_validation,
                y_pred_clasificacion,
                average="macro",
                zero_division=0,
            )
        ],
    },
    index=["Random Forest Classifier"],
)

# Unimos las métricas de la referencia y del modelo
# para facilitar la comparación.
comparacion_validacion = pd.concat(
    [
        resultados_referencia,
        resultados_rf_validacion,
    ]
)

display(
    comparacion_validacion.round(3)
)

,Accuracy,Balanced accuracy,Precision macro,Recall macro,F1 macro
Referencia: clase mayoritaria,0.353,0.333,0.118,0.333,0.174
Random Forest Classifier,0.937,0.936,0.936,0.936,0.936


#### 6.2. Desempeño por clase

Las métricas generales muestran el comportamiento promedio del clasificador, pero no permiten observar si alguna categoría presenta mayores dificultades que las demás.

Por este motivo, calculamos precision, recall y F1 para cada clase por separado. Esto permite identificar si el modelo distingue correctamente los niveles de producción Baja, Media y Alta, y observar cuáles son las categorías que se confunden con mayor frecuencia.

In [13]:
# Calculamos las métricas de cada clase por separado.
reporte_validacion = pd.DataFrame(
    classification_report(
        y_validation,
        y_pred_clasificacion,
        labels=["Baja", "Media", "Alta"],
        output_dict=True,
        zero_division=0,
    )
).T

display(
    reporte_validacion.round(3)
)

,precision,recall,f1-score,support
Baja,0.962,0.941,0.952,5055.000
Media,0.913,0.902,0.908,4404.000
Alta,0.933,0.965,0.949,4861.000
accuracy,0.937,0.937,0.937,0.937
macro avg,0.936,0.936,0.936,14320.000
weighted avg,0.937,0.937,0.937,14320.000


#### 6.3. Matriz de confusión

La matriz de confusión permite comparar la clase real con la clase predicha por el modelo.

Las filas representan las categorías reales y las columnas representan las categorías predichas. Los valores de la diagonal corresponden a las clasificaciones correctas, mientras que los valores fuera de la diagonal muestran los errores.

Esta matriz permite observar, por ejemplo, si el modelo confunde principalmente categorías cercanas, como Baja con Media o Media con Alta.

In [14]:
# Construimos la matriz de confusión utilizando
# un orden fijo para las tres categorías.
clases_ordenadas = ["Baja", "Media", "Alta"]

matriz_confusion_validacion = pd.DataFrame(
    confusion_matrix(
        y_validation,
        y_pred_clasificacion,
        labels=clases_ordenadas,
    ),
    index=[
        "Real: Baja",
        "Real: Media",
        "Real: Alta",
    ],
    columns=[
        "Predicha: Baja",
        "Predicha: Media",
        "Predicha: Alta",
    ],
)

display(matriz_confusion_validacion)

,Predicha: Baja,Predicha: Media,Predicha: Alta
Real: Baja,4758,246,51
Real: Media,147,3973,284
Real: Alta,39,131,4691


### 7. Validación cruzada temporal y búsqueda de hiperparámetros

Hasta aquí evaluamos el clasificador sobre una única ventana de validación (enero a marzo de 2026). Para que la evaluación no dependa de un solo período, y para elegir los hiperparámetros de forma sistemática en lugar de fijarlos manualmente, combinamos:

- una **validación cruzada temporal** con varios pliegues cronológicos;
- una búsqueda de hiperparámetros con **`GridSearchCV`** sobre esos mismos pliegues.

#### 7.1. Pliegues temporales

No usamos `TimeSeriesSplit` directamente porque divide las filas según su posición: como cada mes contiene miles de pozos, un corte por posición podría dejar parte de un mes en entrenamiento y otra parte del mismo mes en validación. En su lugar, construimos los pliegues según el **mes del objetivo**, con el mismo criterio de `TimeSeriesSplit` (ventana de entrenamiento creciente y validación siempre posterior) y los mismos pliegues que usamos en la notebook 04:

| Pliegue | Entrenamiento (objetivos) | Validación (objetivos) |
|---|---|---|
| 1 | febrero a julio de 2025 | agosto y septiembre de 2025 |
| 2 | febrero a septiembre de 2025 | octubre y noviembre de 2025 |
| 3 | febrero a noviembre de 2025 | diciembre de 2025 |

Los pliegues se construyen solamente con el conjunto de entrenamiento. El conjunto de validación (enero a marzo de 2026) y el de test (abril a julio de 2026) no participan de la búsqueda.

Descartamos una validación cruzada agrupada por pozo (`GroupKFold`) porque mezclaría meses: el modelo podría entrenarse con objetivos posteriores a los que se usan para evaluarlo, lo que contradice el criterio temporal que aplicamos en todo el proyecto.

In [15]:
# Definimos los pliegues por mes del objetivo, igual que en la notebook 04.
pliegues_temporales = [
    {"Pliegue": "1", "Entrenamiento hasta": "2025-07-01", "Validación desde": "2025-08-01", "Validación hasta": "2025-09-01"},
    {"Pliegue": "2", "Entrenamiento hasta": "2025-09-01", "Validación desde": "2025-10-01", "Validación hasta": "2025-11-01"},
    {"Pliegue": "3", "Entrenamiento hasta": "2025-11-01", "Validación desde": "2025-12-01", "Validación hasta": "2025-12-01"},
]

# GridSearchCV acepta una lista de pares (índices de entrenamiento, índices de validación).
# Los índices son posiciones dentro de X_train.
mes_objetivo_train = train["mes_objetivo"].reset_index(drop=True)

pliegues_cv = []
resumen_pliegues = []

for pliegue in pliegues_temporales:
    mascara_entrenamiento = mes_objetivo_train <= pd.Timestamp(pliegue["Entrenamiento hasta"])
    mascara_validacion = mes_objetivo_train.between(
        pd.Timestamp(pliegue["Validación desde"]),
        pd.Timestamp(pliegue["Validación hasta"]),
    )

    indices_entrenamiento = np.flatnonzero(mascara_entrenamiento.to_numpy())
    indices_validacion = np.flatnonzero(mascara_validacion.to_numpy())

    if len(indices_entrenamiento) == 0 or len(indices_validacion) == 0:
        raise ValueError(f"El pliegue {pliegue['Pliegue']} no tiene filas suficientes.")

    # Confirmamos que la validación sea siempre posterior al entrenamiento.
    if mes_objetivo_train[mascara_entrenamiento].max() >= mes_objetivo_train[mascara_validacion].min():
        raise ValueError(f"El pliegue {pliegue['Pliegue']} no respeta el orden temporal.")

    pliegues_cv.append((indices_entrenamiento, indices_validacion))

    resumen_pliegues.append({
        "Pliegue": pliegue["Pliegue"],
        "Objetivos de entrenamiento": (
            f"{mes_objetivo_train[mascara_entrenamiento].min():%Y-%m} a "
            f"{mes_objetivo_train[mascara_entrenamiento].max():%Y-%m}"
        ),
        "Objetivos de validación": (
            f"{mes_objetivo_train[mascara_validacion].min():%Y-%m} a "
            f"{mes_objetivo_train[mascara_validacion].max():%Y-%m}"
        ),
        "Filas de entrenamiento": len(indices_entrenamiento),
        "Filas de validación": len(indices_validacion),
    })

display(pd.DataFrame(resumen_pliegues).set_index("Pliegue"))

,Objetivos de entrenamiento,Objetivos de validación,Filas de entrenamiento,Filas de validación
Pliegue,,,,
1,2025-02 a 2025-07,2025-08 a 2025-09,26323,9118
2,2025-02 a 2025-09,2025-10 a 2025-11,35441,9307
3,2025-02 a 2025-11,2025-12 a 2025-12,44748,4704


#### 7.2. Búsqueda de hiperparámetros con `GridSearchCV`

Buscamos la mejor combinación de tres hiperparámetros del Random Forest:

- **`max_depth`**: profundidad máxima de cada árbol. `None` permite árboles sin límite; valores menores generan árboles más simples y menos propensos al sobreajuste.
- **`min_samples_leaf`**: cantidad mínima de registros en cada hoja.
- **`max_features`**: proporción de variables consideradas en cada división.

Cada combinación se entrena y evalúa en los tres pliegues temporales. Registramos balanced accuracy y F1 macro, y elegimos la configuración con el mejor **F1 macro promedio**, porque considera por igual el desempeño de las tres clases.

El preprocesamiento forma parte del pipeline, por lo que en cada pliegue las medianas, las categorías y los indicadores de faltantes se aprenden solamente con los meses de entrenamiento de ese pliegue.

In [16]:
import time

from sklearn.model_selection import GridSearchCV

grilla_clasificacion = {
    "modelo__max_depth": [None, 20, 10],
    "modelo__min_samples_leaf": [5, 20],
    "modelo__max_features": [0.7, "sqrt"],
}

busqueda_clasificacion = GridSearchCV(
    estimator=clone(modelo_clasificacion),
    param_grid=grilla_clasificacion,
    scoring={
        "balanced_accuracy": "balanced_accuracy",
        "f1_macro": "f1_macro",
    },
    refit="f1_macro",
    cv=pliegues_cv,
    # El Random Forest ya usa todos los núcleos (n_jobs=-1).
    n_jobs=1,
    return_train_score=False,
)

cantidad_combinaciones = int(np.prod([len(valores) for valores in grilla_clasificacion.values()]))
print(
    f"Evaluando {cantidad_combinaciones} combinaciones en {len(pliegues_cv)} pliegues "
    f"({cantidad_combinaciones * len(pliegues_cv)} entrenamientos)...",
    flush=True,
)

inicio_busqueda = time.time()
busqueda_clasificacion.fit(X_train, y_train)
print(f"Búsqueda finalizada en {(time.time() - inicio_busqueda) / 60:.1f} minutos.")

resultados_busqueda = pd.DataFrame(busqueda_clasificacion.cv_results_)

tabla_busqueda = pd.DataFrame({
    "max_depth": resultados_busqueda["param_modelo__max_depth"].astype(str),
    "min_samples_leaf": resultados_busqueda["param_modelo__min_samples_leaf"],
    "max_features": resultados_busqueda["param_modelo__max_features"].astype(str),
    "F1 macro medio": resultados_busqueda["mean_test_f1_macro"],
    "F1 macro desvío": resultados_busqueda["std_test_f1_macro"],
    "Balanced accuracy media": resultados_busqueda["mean_test_balanced_accuracy"],
    "Ranking F1": resultados_busqueda["rank_test_f1_macro"],
}).sort_values("Ranking F1")

display(tabla_busqueda.round(4).reset_index(drop=True))

mejores_parametros = busqueda_clasificacion.best_params_
print("Mejor configuración:", mejores_parametros)

Evaluando 12 combinaciones en 3 pliegues (36 entrenamientos)...


Búsqueda finalizada en 8.4 minutos.


,max_depth,min_samples_leaf,max_features,F1 macro medio,F1 macro desvío,Balanced accuracy media,Ranking F1
0,NaN,20,0.7,0.9349,0.0062,0.9347,1
1,20,20,0.7,0.9349,0.0064,0.9347,2
2,NaN,5,0.7,0.9345,0.0066,0.9344,3
3,20,5,0.7,0.9344,0.0064,0.9344,4
4,NaN,20,sqrt,0.9344,0.0054,0.9343,5
5,NaN,5,sqrt,0.9343,0.0044,0.9342,6
6,10,20,0.7,0.9342,0.0056,0.9340,7
7,20,5,sqrt,0.9342,0.0047,0.9341,8
8,20,20,sqrt,0.9341,0.0052,0.9340,9
9,10,5,0.7,0.9339,0.0057,0.9338,10


Mejor configuración: {'modelo__max_depth': None, 'modelo__max_features': 0.7, 'modelo__min_samples_leaf': 20}


#### 7.3. Resultados por pliegue de la mejor configuración

Para comprobar que la mejora no depende de un único período, mostramos el desempeño de la mejor configuración en cada pliegue y lo comparamos con la referencia de clase mayoritaria, ajustada con los mismos meses de entrenamiento de cada pliegue.

In [17]:
from sklearn.dummy import DummyClassifier

indice_mejor = busqueda_clasificacion.best_index_
resultados_por_pliegue = []

for numero, (indices_entrenamiento, indices_validacion) in enumerate(pliegues_cv):
    y_entrenamiento_pliegue = y_train.iloc[indices_entrenamiento]
    y_validacion_pliegue = y_train.iloc[indices_validacion]

    referencia_pliegue = DummyClassifier(strategy="most_frequent")
    referencia_pliegue.fit(X_train.iloc[indices_entrenamiento], y_entrenamiento_pliegue)
    prediccion_referencia = referencia_pliegue.predict(X_train.iloc[indices_validacion])

    resultados_por_pliegue.append({
        "Pliegue": pliegues_temporales[numero]["Pliegue"],
        "Objetivos de validación": resumen_pliegues[numero]["Objetivos de validación"],
        "Filas de validación": len(indices_validacion),
        "Balanced accuracy referencia": balanced_accuracy_score(y_validacion_pliegue, prediccion_referencia),
        "Balanced accuracy Random Forest": resultados_busqueda.loc[indice_mejor, f"split{numero}_test_balanced_accuracy"],
        "F1 macro referencia": f1_score(y_validacion_pliegue, prediccion_referencia, average="macro", zero_division=0),
        "F1 macro Random Forest": resultados_busqueda.loc[indice_mejor, f"split{numero}_test_f1_macro"],
    })

tabla_pliegues = pd.DataFrame(resultados_por_pliegue).set_index("Pliegue")
display(tabla_pliegues.round(3))

columnas_metricas = [columna for columna in tabla_pliegues.columns if "Random Forest" in columna or "referencia" in columna]
print("Promedio y desvío entre pliegues:")
display(tabla_pliegues[columnas_metricas].agg(["mean", "std"]).round(3))

,Objetivos de validación,Filas de validación,Balanced accuracy referencia,Balanced accuracy Random Forest,F1 macro referencia,F1 macro Random Forest
Pliegue,,,,,,
1,2025-08 a 2025-09,9118,0.333,0.933,0.179,0.933
2,2025-10 a 2025-11,9307,0.333,0.928,0.177,0.928
3,2025-12 a 2025-12,4704,0.333,0.943,0.171,0.943


Promedio y desvío entre pliegues:


,Balanced accuracy referencia,Balanced accuracy Random Forest,F1 macro referencia,F1 macro Random Forest
mean,0.333,0.935,0.176,0.935
std,0.000,0.007,0.004,0.008


#### 7.4. Comparación en el conjunto de validación

Por último, comparamos en el conjunto de validación (enero a marzo de 2026) la configuración inicial, entrenada en la sección 6, con la configuración elegida por `GridSearchCV`. `GridSearchCV` ya reentrenó la mejor configuración con todo el conjunto de entrenamiento (`refit`), por lo que podemos usarla directamente.

In [18]:
modelo_clasificacion_busqueda = busqueda_clasificacion.best_estimator_

predicciones_validacion = {
    "Referencia: clase mayoritaria": y_pred_referencia,
    "Random Forest inicial": y_pred_clasificacion,
    "Random Forest GridSearchCV": modelo_clasificacion_busqueda.predict(X_validation),
}

comparacion_busqueda = pd.DataFrame({
    "Accuracy": {nombre: accuracy_score(y_validation, prediccion) for nombre, prediccion in predicciones_validacion.items()},
    "Balanced accuracy": {nombre: balanced_accuracy_score(y_validation, prediccion) for nombre, prediccion in predicciones_validacion.items()},
    "F1 macro": {nombre: f1_score(y_validation, prediccion, average="macro", zero_division=0) for nombre, prediccion in predicciones_validacion.items()},
})

display(comparacion_busqueda.round(3))

,Accuracy,Balanced accuracy,F1 macro
Referencia: clase mayoritaria,0.353,0.333,0.174
Random Forest inicial,0.937,0.936,0.936
Random Forest GridSearchCV,0.938,0.937,0.937


#### 7.5. Interpretación de la validación cruzada y de la búsqueda

- **El desempeño es estable entre períodos.** La mejor configuración obtuvo un F1 macro de 0,933, 0,928 y 0,943 en los tres pliegues (promedio 0,935, desvío 0,008). La referencia de clase mayoritaria se mantuvo en un F1 macro cercano a 0,18 en todos los pliegues. Por lo tanto, la ventaja del Random Forest no depende de una única ventana de evaluación.
- **El modelo es poco sensible a los hiperparámetros.** Las 12 combinaciones evaluadas obtuvieron un F1 macro promedio entre 0,934 y 0,935, una diferencia menor que la variación entre pliegues. Esto indica que el desempeño proviene principalmente de las variables predictoras y no de un ajuste fino de los hiperparámetros.
- **Configuración seleccionada.** `GridSearchCV` eligió árboles sin límite de profundidad (`max_depth=None`), al menos 20 registros por hoja (`min_samples_leaf=20`) y el 70 % de las variables en cada división (`max_features=0.7`). Las hojas más grandes generan árboles algo más simples que la configuración inicial, con el mismo desempeño.
- **Validación.** En enero a marzo de 2026, la configuración elegida obtuvo un F1 macro de 0,937, prácticamente igual al del Random Forest inicial (0,936).

### 8. Selección y evaluación final del clasificador

Seleccionamos como clasificador final la configuración elegida por `GridSearchCV` en la sección 7, a partir del F1 macro promedio en los tres pliegues temporales. La elección se realizó utilizando únicamente el conjunto de entrenamiento. El conjunto de test se mantiene reservado para evaluar el desempeño final sobre objetivos posteriores, correspondientes a abril–julio de 2026.

Para aprovechar la información disponible antes del test, ajustamos nuevamente el clasificador utilizando conjuntamente entrenamiento y validación. Esta decisión se toma después de seleccionar la configuración y no modifica el conjunto de test.

#### 8.1. Entrenamiento final del clasificador

El clasificador final se entrena con las filas de entrenamiento y validación, cuyos objetivos abarcan desde febrero de 2025 hasta marzo de 2026.

El conjunto de test, con objetivos de abril a julio de 2026, no participa del ajuste. De esta manera, podemos evaluar el modelo sobre un período posterior al utilizado para entrenarlo.

In [19]:
# Unimos entrenamiento y validación después de seleccionar
# la configuración final mediante validación.
X_train_validacion = pd.concat(
    [X_train, X_validation],
    axis=0,
    ignore_index=True,
)

y_train_validacion = pd.concat(
    [y_train, y_validation],
    axis=0,
    ignore_index=True,
)

# Creamos una copia independiente del pipeline seleccionado.
modelo_clasificador_final = clone(modelo_clasificacion)
# Aplicamos la mejor configuración encontrada por GridSearchCV.
modelo_clasificador_final.set_params(**mejores_parametros)
print("Configuración final:", mejores_parametros)

print(
    "Entrenando el clasificador final con entrenamiento "
    "y validación...",
    flush=True,
)

modelo_clasificador_final.fit(
    X_train_validacion,
    y_train_validacion,
)

print("Entrenamiento finalizado.")
print(
    f"Filas utilizadas: {len(X_train_validacion):,}"
)
print(
    f"Clases utilizadas: "
    f"{sorted(y_train_validacion.unique())}"
)

Configuración final: {'modelo__max_depth': None, 'modelo__max_features': 0.7, 'modelo__min_samples_leaf': 20}
Entrenando el clasificador final con entrenamiento y validación...


Entrenamiento finalizado.
Filas utilizadas: 63,772
Clases utilizadas: ['Alta', 'Baja', 'Media']


#### 8.2. Evaluación final en el conjunto de test

Evaluamos el clasificador final sobre el conjunto de test, que contiene los objetivos de abril a julio de 2026.

Calculamos las mismas métricas utilizadas durante la validación para poder comparar los resultados:

- Accuracy;
- Balanced accuracy;
- Precision macro;
- Recall macro;
- F1 macro.

Además, mostramos el desempeño por clase y la matriz de confusión. Como el conjunto de test se utiliza en esta etapa únicamente para informar el resultado final, no se realizarán nuevos ajustes a partir de estas métricas.

In [20]:
# Generamos las predicciones finales sobre test.
y_pred_test_clasificacion = modelo_clasificador_final.predict(
    X_test
)

resultados_test_clasificacion = pd.DataFrame(
    {
        "Accuracy": [
            accuracy_score(
                y_test,
                y_pred_test_clasificacion,
            )
        ],
        "Balanced accuracy": [
            balanced_accuracy_score(
                y_test,
                y_pred_test_clasificacion,
            )
        ],
        "Precision macro": [
            precision_score(
                y_test,
                y_pred_test_clasificacion,
                average="macro",
                zero_division=0,
            )
        ],
        "Recall macro": [
            recall_score(
                y_test,
                y_pred_test_clasificacion,
                average="macro",
                zero_division=0,
            )
        ],
        "F1 macro": [
            f1_score(
                y_test,
                y_pred_test_clasificacion,
                average="macro",
                zero_division=0,
            )
        ],
    },
    index=["Random Forest Classifier final"],
)

display(
    resultados_test_clasificacion.round(3)
)

,Accuracy,Balanced accuracy,Precision macro,Recall macro,F1 macro
Random Forest Classifier final,0.936,0.933,0.935,0.933,0.934


In [21]:
# Evaluamos precision, recall y F1 para cada categoría.
reporte_test_clasificacion = pd.DataFrame(
    classification_report(
        y_test,
        y_pred_test_clasificacion,
        labels=["Baja", "Media", "Alta"],
        output_dict=True,
        zero_division=0,
    )
).T

display(
    reporte_test_clasificacion.round(3)
)

,precision,recall,f1-score,support
Baja,0.947,0.948,0.947,6906.000
Media,0.919,0.886,0.902,5825.000
Alta,0.938,0.965,0.951,7006.000
accuracy,0.936,0.936,0.936,0.936
macro avg,0.935,0.933,0.934,19737.000
weighted avg,0.935,0.936,0.935,19737.000


In [22]:
# Construimos la matriz de confusión final.
matriz_confusion_test = pd.DataFrame(
    confusion_matrix(
        y_test,
        y_pred_test_clasificacion,
        labels=["Baja", "Media", "Alta"],
    ),
    index=[
        "Real: Baja",
        "Real: Media",
        "Real: Alta",
    ],
    columns=[
        "Predicha: Baja",
        "Predicha: Media",
        "Predicha: Alta",
    ],
)

display(matriz_confusion_test)

,Predicha: Baja,Predicha: Media,Predicha: Alta
Real: Baja,6547,272,87
Real: Media,305,5161,359
Real: Alta,65,181,6760


### 9. Inferencia de categorías para agosto de 2026

Como complemento del modelo de regresión, aplicamos el clasificador final al conjunto de inferencia correspondiente a julio de 2026.

El modelo asigna a cada pozo una categoría esperada de producción para agosto de 2026: Baja, Media o Alta. Estas predicciones no se evalúan con métricas porque todavía no conocemos la producción real del mes objetivo.

In [23]:
# Cargamos el conjunto de inferencia generado en la notebook 03.
ruta_inferencia = RUTA_PROCESSED / "inferencia_modelado.csv"

inferencia = pd.read_csv(
    ruta_inferencia,
    parse_dates=["fecha", "mes_objetivo"],
    low_memory=False,
)

columnas_faltantes_inferencia = sorted(
    set(features) - set(inferencia.columns)
)

if columnas_faltantes_inferencia:
    raise KeyError(
        "Faltan variables predictoras en inferencia: "
        f"{columnas_faltantes_inferencia}"
    )

X_inferencia = inferencia[features].copy()

# Aplicamos la misma conversión utilizada durante el entrenamiento.
for columna in features_booleanas:
    X_inferencia[columna] = (
        X_inferencia[columna].astype("float64")
    )

clases_predichas_agosto = (
    modelo_clasificador_final.predict(X_inferencia)
)

resultado_clasificacion_inferencia = inferencia[
    ["idpozo", "sigla", "fecha", "mes_objetivo"]
].copy()

resultado_clasificacion_inferencia[
    "clase_predicha_agosto_2026"
] = clases_predichas_agosto

display(
    resultado_clasificacion_inferencia.head(10)
)

,idpozo,sigla,fecha,mes_objetivo,clase_predicha_agosto_2026
0,3640,PPC.Nq.EC-4,2026-07-01,2026-08-01,Baja
1,8043,YPF.Nq.FP.x-2,2026-07-01,2026-08-01,Baja
2,10073,CSJ.RN.LMa.a-3,2026-07-01,2026-08-01,Media
3,11191,Q.SC.ACI-21,2026-07-01,2026-08-01,Media
4,11641,TPT.Nq.PS-1001,2026-07-01,2026-08-01,Media
5,11642,TPT.Nq.PS-1002,2026-07-01,2026-08-01,Baja
6,11643,YPF.Nq.PS.x-2,2026-07-01,2026-08-01,Baja
7,34833,PAE.Nq.LA.xp-89,2026-07-01,2026-08-01,Baja
8,42900,YPF.Nq.BP.x-2,2026-07-01,2026-08-01,Baja
9,42905,YPF.Nq.BP.a-7,2026-07-01,2026-08-01,Baja


In [24]:
resumen_clases_inferencia = (
    resultado_clasificacion_inferencia[
        "clase_predicha_agosto_2026"
    ]
    .value_counts()
    .rename_axis("Clase predicha")
    .reset_index(name="Registros")
)

resumen_clases_inferencia["Porcentaje (%)"] = (
    resumen_clases_inferencia["Registros"]
    / len(resultado_clasificacion_inferencia)
    * 100
)

display(
    resumen_clases_inferencia.round(2)
)

,Clase predicha,Registros,Porcentaje (%)
0,Alta,1919,37.91
1,Baja,1677,33.13
2,Media,1466,28.96


In [25]:
ruta_clasificacion_inferencia = (
    RUTA_PROCESSED
    / "predicciones_clasificacion_agosto_2026.csv"
)

resultado_clasificacion_inferencia.to_csv(
    ruta_clasificacion_inferencia,
    index=False,
)

print(
    f"Archivo guardado en: {ruta_clasificacion_inferencia}"
)

Archivo guardado en: /Users/julianvarelasilva/Downloads/ypf-data/data/processed/predicciones_clasificacion_agosto_2026.csv


### 10. Conclusión del modelo de clasificación

El Random Forest Classifier logró clasificar la producción del mes siguiente en las categorías Baja, Media y Alta con un desempeño elevado.

En el conjunto de test obtuvo:

- Accuracy: 0,936;
- Balanced accuracy: 0,933;
- Precision macro: 0,935;
- Recall macro: 0,933;
- F1 macro: 0,934.

La evaluación no descansa sobre una única ventana. En la validación cruzada temporal, con tres pliegues cronológicos entre agosto y diciembre de 2025, el F1 macro fue de 0,935 en promedio, con un desvío de 0,008. En validación (enero a marzo de 2026) fue de 0,937 y en test (abril a julio de 2026) de 0,934. Esta consistencia entre períodos indica que el modelo conserva su comportamiento al evaluarse sobre meses posteriores y no presenta una caída importante por sobreajuste.

Los hiperparámetros se seleccionaron con `GridSearchCV` sobre los pliegues temporales, en lugar de fijarlos manualmente. Las 12 combinaciones evaluadas de `max_depth`, `min_samples_leaf` y `max_features` obtuvieron resultados muy similares, por lo que el modelo es robusto a estas decisiones. La configuración final usa `max_depth=None`, `min_samples_leaf=20` y `max_features=0.7`.

El tratamiento de faltantes se alineó con el de la notebook 04: las categorías faltantes se representan como `"Sin dato"` y las variables numéricas imputadas tienen columnas indicadoras. De esta manera, el modelo distingue a los pozos sin información de fracturación, que tienen una proporción mucho mayor de producción Baja.

La clase Alta obtuvo el mejor recall, con 0,965, mientras que la clase Media presentó el desempeño más bajo, con un recall de 0,886. Esto indica que los casos intermedios son más difíciles de distinguir, probablemente porque se encuentran cerca de los límites que separan las categorías.

La matriz de confusión mostró que los errores se concentran principalmente entre categorías consecutivas, como Baja y Media o Media y Alta. En cambio, las confusiones entre Baja y Alta fueron poco frecuentes.

La referencia de clase mayoritaria obtuvo un F1 macro de 0,174 y una balanced accuracy de 0,333, por lo que el clasificador final aporta una mejora clara y utiliza información relevante de las variables del pozo y de su historial productivo.

Este modelo de clasificación se presenta como un complemento del modelo principal de regresión. La regresión permite estimar la cantidad de petróleo en metros cúbicos, mientras que la clasificación permite resumir esa producción futura en niveles más fáciles de interpretar.